# NER Task: Tokenizer Comparison

This notebook adds a Named Entity Recognition (NER)-oriented evaluation to the domain-specific tokenization project.

## Goal
Compare baseline and custom tokenizers on a downstream NER setting by measuring how they preserve entity structure in a labeled NER dataset.

## Why this is useful
For NER, tokenization quality strongly affects label alignment and entity boundary consistency. A tokenizer that over-fragments entities can make sequence labeling harder.


In [8]:
import os
import numpy as np
import pandas as pd
from datasets import load_dataset
from tokenizers import Tokenizer
from transformers import BertTokenizer, GPT2Tokenizer


In [9]:
TOKENIZERS_DIR = "../tokenizers"

TOKENIZER_CONFIGS = {
    "bert_base": {"type": "bert", "name": "bert-base-uncased"},
    "biobert": {"type": "bert", "name": "dmis-lab/biobert-base-cased-v1.1"},
    "gpt2": {"type": "gpt2", "name": "gpt2"},
    "biomedical_bpe_30k": {
        "type": "custom_bpe",
        "path": os.path.join(TOKENIZERS_DIR, "biomedical_bpe_30k.json"),
    },
    "biomedical_bpe_50k": {
        "type": "custom_bpe",
        "path": os.path.join(TOKENIZERS_DIR, "biomedical_bpe_50k.json"),
    },
    "biomedical_bpe_100k": {
        "type": "custom_bpe",
        "path": os.path.join(TOKENIZERS_DIR, "biomedical_bpe_100k.json"),
    },
}


def build_tokenizer(config):
    if config["type"] == "bert":
        tok = BertTokenizer.from_pretrained(config["name"])
        return {"encode": lambda text: tok.tokenize(text), "unk_token": tok.unk_token}

    if config["type"] == "gpt2":
        tok = GPT2Tokenizer.from_pretrained(config["name"])
        return {"encode": lambda text: tok.tokenize(text), "unk_token": tok.unk_token}

    if config["type"] == "custom_bpe":
        tok = Tokenizer.from_file(config["path"])
        return {"encode": lambda text: tok.encode(text).tokens, "unk_token": "[UNK]"}

    raise ValueError(f"Unknown tokenizer type: {config['type']}")


In [10]:
# JNLPBA is a biomedical NER benchmark (GENE/PROTEIN, DNA, RNA, CELL_TYPE, CELL_LINE).
# We use an external Hub dataset copy compatible with recent `datasets` releases.
raw_ds = load_dataset("siddharthtumre/jnlpba-split")

MAX_TRAIN_ROWS = 4000
MAX_VALID_ROWS = 1000

train_ds = raw_ds["train"].select(range(min(MAX_TRAIN_ROWS, len(raw_ds["train"]))))
valid_ds = raw_ds["validation"].select(range(min(MAX_VALID_ROWS, len(raw_ds["validation"]))))

print("Train subset:", len(train_ds))
print("Validation subset:", len(valid_ds))
print("NER tag id range:", int(min(min(x["ner_tags"]) for x in train_ds)), "..", int(max(max(x["ner_tags"]) for x in train_ds)))


Train subset: 4000
Validation subset: 1000
NER tag id range: 0 .. 10


In [11]:
def decode_entities(tag_ids):
    """
    Decode entity spans from integer BIO-style tags without relying on dataset-specific label names.
    Assumption used by many corpora: 0 = O, positive ids form B/I pairs.
    """
    entities = []
    i = 0
    n = len(tag_ids)

    def entity_type_id(tag_id):
        # 1/2 -> type 1, 3/4 -> type 2, etc.
        return (tag_id + 1) // 2

    while i < n:
        tag = tag_ids[i]
        if tag <= 0:
            i += 1
            continue

        cur_type = entity_type_id(tag)
        start = i
        i += 1

        while i < n and tag_ids[i] > 0 and entity_type_id(tag_ids[i]) == cur_type:
            i += 1

        entities.append((start, i))

    return entities


def safe_tokenize(encode_fn, text):
    pieces = encode_fn(text)
    if pieces is None:
        return []
    return [p for p in pieces if p]


In [12]:
def evaluate_tokenizer_on_ner(dataset, tokenizer_bundle):
    encode_fn = tokenizer_bundle["encode"]
    unk_token = tokenizer_bundle["unk_token"]

    total_words = 0
    total_word_subtokens = 0
    total_word_unk = 0

    total_entities = 0
    total_entity_subtokens = 0
    entity_single_piece = 0
    entity_fragmented = 0
    entity_has_unk = 0

    for sample in dataset:
        words = sample["tokens"]
        tags = sample["ner_tags"]

        for word in words:
            word_pieces = safe_tokenize(encode_fn, word)
            if not word_pieces:
                continue
            total_words += 1
            total_word_subtokens += len(word_pieces)
            total_word_unk += sum(piece == unk_token for piece in word_pieces)

        entities = decode_entities(tags)
        for start, end in entities:
            entity_text = " ".join(words[start:end])
            entity_pieces = safe_tokenize(encode_fn, entity_text)
            if not entity_pieces:
                continue

            total_entities += 1
            piece_count = len(entity_pieces)
            total_entity_subtokens += piece_count

            if piece_count == 1:
                entity_single_piece += 1
            if piece_count > 1:
                entity_fragmented += 1
            if any(piece == unk_token for piece in entity_pieces):
                entity_has_unk += 1

    eps = 1e-9
    return {
        "avg_subtokens_per_word": total_word_subtokens / (total_words + eps),
        "unk_rate_word_level": total_word_unk / (total_word_subtokens + eps),
        "avg_subtokens_per_entity": total_entity_subtokens / (total_entities + eps),
        "entity_single_piece_rate": entity_single_piece / (total_entities + eps),
        "entity_fragmentation_rate": entity_fragmented / (total_entities + eps),
        "entity_unk_rate": entity_has_unk / (total_entities + eps),
        "entity_count": total_entities,
    }


In [13]:
results = []

for name, config in TOKENIZER_CONFIGS.items():
    bundle = build_tokenizer(config)

    train_metrics = evaluate_tokenizer_on_ner(train_ds, bundle)
    valid_metrics = evaluate_tokenizer_on_ner(valid_ds, bundle)

    row = {"tokenizer": name}
    row.update({f"train_{k}": v for k, v in train_metrics.items()})
    row.update({f"valid_{k}": v for k, v in valid_metrics.items()})
    results.append(row)

results_df = pd.DataFrame(results)
results_df


,tokenizer,train_avg_subtokens_per_word,train_unk_rate_word_level,train_avg_subtokens_per_entity,train_entity_single_piece_rate,train_entity_fragmentation_rate,train_entity_unk_rate,train_entity_count,valid_avg_subtokens_per_word,valid_unk_rate_word_level,valid_avg_subtokens_per_entity,valid_entity_single_piece_rate,valid_entity_fragmentation_rate,valid_entity_unk_rate,valid_entity_count
0,bert_base,1.457393,0.0,4.694575,0.028371,0.971629,0.0,10821,1.436713,0.0,4.663928,0.033981,0.966019,0.0,2678
1,biobert,1.551605,0.0,5.207467,0.013492,0.986508,0.0,10821,1.523613,0.0,5.165049,0.010829,0.989171,0.0,2678
2,gpt2,1.648071,0.0,4.743369,0.017004,0.982996,0.0,10821,1.624781,0.0,4.696415,0.020911,0.979089,0.0,2678
3,biomedical_bpe_30k,1.221574,0.0,3.572221,0.120414,0.879586,0.0,10821,1.211604,0.0,3.570575,0.115011,0.884989,0.0,2678
4,biomedical_bpe_50k,1.200595,0.0,3.470474,0.143055,0.856945,0.0,10821,1.191157,0.0,3.477595,0.131068,0.868932,0.0,2678
5,biomedical_bpe_100k,1.200595,0.0,3.470474,0.143055,0.856945,0.0,10821,1.191157,0.0,3.477595,0.131068,0.868932,0.0,2678


In [14]:
# A simple aggregate score (higher is better).
# It rewards compact entities and penalizes unknown tokens.
rank_df = results_df.copy()
rank_df["ner_tokenization_score"] = (
    0.50 * (1.0 - rank_df["valid_entity_fragmentation_rate"])
    + 0.30 * rank_df["valid_entity_single_piece_rate"]
    + 0.20 * (1.0 - rank_df["valid_entity_unk_rate"])
)

rank_columns = [
    "tokenizer",
    "ner_tokenization_score",
    "valid_avg_subtokens_per_word",
    "valid_avg_subtokens_per_entity",
    "valid_entity_single_piece_rate",
    "valid_entity_fragmentation_rate",
    "valid_entity_unk_rate",
]

rank_df = rank_df[rank_columns].sort_values("ner_tokenization_score", ascending=False).reset_index(drop=True)
rank_df


,tokenizer,ner_tokenization_score,valid_avg_subtokens_per_word,valid_avg_subtokens_per_entity,valid_entity_single_piece_rate,valid_entity_fragmentation_rate,valid_entity_unk_rate
0,biomedical_bpe_50k,0.304854,1.191157,3.477595,0.131068,0.868932,0.0
1,biomedical_bpe_100k,0.304854,1.191157,3.477595,0.131068,0.868932,0.0
2,biomedical_bpe_30k,0.292009,1.211604,3.570575,0.115011,0.884989,0.0
3,bert_base,0.227184,1.436713,4.663928,0.033981,0.966019,0.0
4,gpt2,0.216729,1.624781,4.696415,0.020911,0.979089,0.0
5,biobert,0.208663,1.523613,5.165049,0.010829,0.989171,0.0


## Interpretation Guide

- Lower `avg_subtokens_per_entity` usually means better entity compactness.
- Higher `entity_single_piece_rate` means more entities are preserved as single units.
- Lower `entity_fragmentation_rate` means easier BIO alignment for NER models.
- Lower `entity_unk_rate` means better lexical coverage.

These metrics do not replace full NER model training, but they provide a lightweight and reproducible proxy for tokenizer suitability in NER pipelines.
